In [ ]:
# Exploratory Data Analysis
# Only run once
import sys
import os

project_root = os.path.abspath('../')
os.chdir(project_root)

src_path = os.path.abspath(os.path.join(os.getcwd(), 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Setup

In [ ]:
from load_data import DataLoader
from preprocess import Preprocessor, FeatureEngineering
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

dl = DataLoader()
pre = Preprocessor()
fe = FeatureEngineering()

In [ ]:
train = dl.load_data('', 'data/raw')
test = dl.load_data('', 'data/raw')

train_df = train.copy()
test_df = test.copy()

In [ ]:
num_cols_real = train.drop(columns=['id', 'target']).select_dtypes(include=['float64', 'int64']).columns
cat_cols_real = train.drop(columns=['id', 'target']).select_dtypes(include=['object', 'category']).columns

num_cols = train_df.drop(columns=['id', 'target']).select_dtypes(include=['float64', 'int64']).columns
cat_cols = train_df.drop(columns=['id', 'target']).select_dtypes(include=['object', 'category']).columns    

# EDA

## Info

In [ ]:
dl.data_info(train)

In [ ]:
dl.preview_data(train)

In [ ]:
dl.data_info(test)

In [ ]:
dl.preview_data(test)

## Missing value

In [ ]:
def show_missing(df, name="data"):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if not missing.empty:
        print(f"Missing values in {name}:")
        display(missing)
        missing.plot(kind='bar', title=f'Missing Values in {name}')
        plt.show()
    else:
        print(f"No missing values found in {name}.")

In [ ]:
show_missing(train, "train")
show_missing(test, "test")

## Data distribution

In [ ]:
n_cols = 2
n_rows = -(-len(num_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(train_df, x=col, kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

for i in range(len(num_cols), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
plt.show()

In [ ]:
n_cols = 2
n_rows = -(-len(cat_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(train_df, x=col, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    # axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].set_ylabel('Frequency')

for i in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
plt.show()

## Outlier

In [ ]:
n_cols = 2
n_rows = -(-len(num_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(train_df, y=col, ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_ylabel(col)
    
for i in range(len(num_cols), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
plt.show()

## Feature-target relationship analysis

### Numerical and target

In [ ]:
n_cols = 2
n_rows = -(-len(num_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.violinplot(train, x='target', y=col, ax=axes[i])
    axes[i].set_title(f'Violin plot of {col} by target')
    axes[i].set_xlabel('y')
    axes[i].set_ylabel(col)
    
for i in range(len(num_cols), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
plt.show()

### Categorical and target

In [ ]:
for col in cat_cols:
    ct = pd.crosstab(train_df[col], train_df['target'], normalize='index')
    ct.plot(kind='bar', stacked=True, figsize=(6, 4))
    plt.title(f'Proportion of target by {col}')
    plt.xlabel(col)
    plt.ylabel('Proportion')
    plt.legend(title='target', loc='upper right')
    plt.tight_layout()
    plt.show()

## Pairplot matrix

In [ ]:
pairplot_cols = num_cols.tolist() + ['target']

sns.pairplot(train[pairplot_cols], hue='target', diag_kind='kde', palette="Set1", corner=True)
plt.suptitle('Pairplot of Numerical Features with Target', y=1.02)
plt.show()

## Multicolinearity analysis

In [ ]:
train_encoded = pre.label_encode(train_df)
train_corr = train_encoded.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(train_corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Heatmap of Encoded Features')
plt.show()

In [ ]:
def analyze_correlation(corr_matrix, top_n=5):
    corr_pairs = (
        corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
    top_pos = corr_pairs.sort_values(by='Correlation', ascending=False).head(top_n)
    top_neg = corr_pairs.sort_values(by='Correlation').head(top_n)
    print("Top positively correlated feature pairs:")
    display(top_pos)
    print("\nTop negatively correlated feature pairs:")
    display(top_neg)

analyze_correlation(train_corr, top_n=5)

## Skewness and kurtosis analysis

In [ ]:
skew_kurt = pd.DataFrame({
    'skew': train[num_cols].skew(),
    'kurtosis': train[num_cols].kurt()
})

display(skew_kurt)

skew_kurt.plot(kind='bar', subplots=True, layout=(2, 1), figsize=(10, 6), title=['Skewness', 'Kurtosis'])
plt.tight_layout()
plt.show()

## Unique values and category frequency analysis

In [ ]:
for col in cat_cols:
    print(f"Feature: {col}")
    print(f"  Unique category count: {train[col].nunique()}")
    top_cat = train[col].value_counts().idxmax()
    top_freq = train[col].value_counts().max()
    print(f"  Most frequent categories: {top_cat} ({top_freq} data)")
    print("-" * 45)

## Constant value analysis

In [ ]:
constant_cols = [col for col in train_df.columns if train_df[col].nunique() == 1]
if constant_cols:
    print("Columns with constant values (only one unique value):")
    for col in constant_cols:
        print(f"- {col}: {train[col].unique()[0]}")
else:
    print("There are no columns with constant values.")